# 06 · Two-stage model (optional phase)

**Idea:** points = minutes × points per minute. Fit two models:
- **Model A** predicts **minutes**.
- **Model B** predicts **points per minute**, weighted by minutes so 2-minute garbage-time games don't dominate.
- Prediction = A × B.

Why try it: Phase 2 showed minutes drive most of the variation in points (r = 0.74). Splitting them out could let each part be modelled better.

Bar to beat: one-stage linear regression, walk-forward test MAE **4.48 / 4.53 / 4.50**.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from fbml.config import ROOT

PROCESSED = ROOT / "nba" / "data" / "processed"
features = pd.read_parquet(PROCESSED / "features.parquet")

ID_COLS = ['season', 'player_id', 'player', 'team', 'opponent', 'game_id', 'date']
TARGET = 'points'
FEATURES = [c for c in features.columns if c not in ID_COLS + [TARGET, 'minutes']]

model_df = features.dropna(subset=FEATURES).copy()
model_df['pts_per_min'] = model_df['points'] / model_df['minutes']   # minutes > 0 always (0-minute rows dropped in Phase 1)


def lin_reg():
    return make_pipeline(StandardScaler(), LinearRegression())

print(model_df.shape)

In [ ]:
# Walk-forward: one-stage vs two-stage, plus an "oracle" that is given tonight's TRUE minutes
rows = []
for test_season in [22023, 22024, 22025]:
    tr = model_df[model_df['season'] < test_season]
    te = model_df[model_df['season'] == test_season]

    one_stage = lin_reg().fit(tr[FEATURES], tr[TARGET])

    minutes_model = lin_reg().fit(tr[FEATURES], tr['minutes'])                        # model A
    rate_model = lin_reg().fit(tr[FEATURES], tr['pts_per_min'],
                               linearregression__sample_weight=tr['minutes'])        # model B
    minutes_hat = minutes_model.predict(te[FEATURES]).clip(min=0)
    rate_hat = rate_model.predict(te[FEATURES]).clip(min=0)

    rows.append({
        'test_season': test_season,
        'one_stage': mean_absolute_error(te[TARGET], one_stage.predict(te[FEATURES])),
        'two_stage': mean_absolute_error(te[TARGET], minutes_hat * rate_hat),
        'minutes_mae': mean_absolute_error(te['minutes'], minutes_hat),
        # NOT usable for betting (true minutes are only known after the game): shows how much minutes matter
        'oracle_true_minutes': mean_absolute_error(te[TARGET], te['minutes'] * rate_hat),
    })

walk = pd.DataFrame(rows).set_index('test_season')
print(walk.round(3))
print()
print("average:", walk.mean().round(3).to_dict())

In [ ]:
# Can a better minutes model help? XGBoost for model A (early stopping on 2024-25), linear rate model B
train = model_df[model_df['season'] < 22024]
valid = model_df[model_df['season'] == 22024]
test = model_df[model_df['season'] == 22025]
train_valid = pd.concat([train, valid])

minutes_xgb = XGBRegressor(n_estimators=2000, max_depth=3, learning_rate=0.03, min_child_weight=50,
                           subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=100,
                           eval_metric='mae', random_state=42)
minutes_xgb.fit(train[FEATURES], train['minutes'], eval_set=[(valid[FEATURES], valid['minutes'])], verbose=0)
minutes_lr = lin_reg().fit(train_valid[FEATURES], train_valid['minutes'])
rate_lr = lin_reg().fit(train_valid[FEATURES], train_valid['pts_per_min'],
                        linearregression__sample_weight=train_valid['minutes'])

rate_hat = rate_lr.predict(test[FEATURES]).clip(min=0)
for name, m in [('linear', minutes_lr), ('xgboost', minutes_xgb)]:
    minutes_hat = m.predict(test[FEATURES]).clip(min=0)
    print(f"minutes model {name:8}: minutes MAE {mean_absolute_error(test['minutes'], minutes_hat):.3f}"
          f" | points MAE {mean_absolute_error(test[TARGET], minutes_hat * rate_hat):.3f}")

In [ ]:
# Where does the one-stage model go wrong? Error by "minutes surprise" (tonight's minutes vs last-10 average)
one_stage = lin_reg().fit(train_valid[FEATURES], train_valid[TARGET])
abs_err = (test[TARGET] - one_stage.predict(test[FEATURES])).abs()
surprise = test['minutes'] - test['minutes_last10']
buckets = pd.cut(surprise, [-60, -10, -5, 5, 10, 60],
                 labels=['10+ fewer', '5-10 fewer', 'within 5', '5-10 more', '10+ more'])
abs_err.groupby(buckets, observed=True).agg(points_mae='mean', games='size').round(2)

## Phase 6 summary
- **Two-stage = one-stage.** Linear minutes × linear rate gives 4.484 / 4.526 / 4.501 vs one-stage 4.482 / 4.526 / 4.499. With linear parts, the split adds nothing.
- **XGBoost for minutes** predicts minutes slightly better (4.58 vs 4.66 MAE) but points barely move.
- **The real finding: minutes are the bottleneck.** Given tonight's *true* minutes, the same rate model would score **3.71–3.77**, about 0.75 points better than anything we have. Errors are 4.15 when minutes land within 5 of normal, and **5.9** when they're 10+ off.
- We predict minutes to within about 4.7 minutes on average. Better pre-game minutes info is the biggest lever left:
  - **injury reports** (who's out, minutes restrictions);
  - **confirmed starters**;
  - **the game spread** (blowout risk means starters sit in the 4th quarter).
- **Decision:** keep the one-stage linear regression. Revisit the two-stage model once live injury and spread features exist.